In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier

In [6]:
def base_pipeline(scaler, model):
    return Pipeline([
        ('scaler', scaler),
        ('model', model)
    ])

def fit_pipeline(pipeline, X_train, y_train):
    return pipeline.fit(X_train, y_train)

def test_pipeline(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)

    return pd.Series({
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro'),
        'recall': recall_score(y_test, y_pred, average='macro')
    })

def save_results(model_name, scaler_name, metrics):
    return {
        'model': model_name,
        'scaler': scaler_name,
        **metrics.to_dict()
    }

def assemble_results(results):
    test_results_df = pd.DataFrame(results).sort_values(['model', 'scaler']).reset_index(drop=True)
    results_by_model_df = test_results_df.set_index(['model', 'scaler']).sort_index()
    results_by_scaler_df = test_results_df.set_index(['scaler', 'model']).sort_index()

    return results_by_model_df, results_by_scaler_df

In [9]:
models = [RandomForestClassifier, LogisticRegression, GaussianNB, SVC, KNeighborsClassifier]
scalers = [MinMaxScaler, StandardScaler]
cols_with_zeros = ['Rash', 'Platelet_Count', 'WBC_Count']
categorical_cols = ["AreaType", "HouseType", "Joint_Pain", "Gender"]

def run_pipeline(X, y, imputer=None, with_logs=False):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
    if imputer is not None:
        # iterate over the columns with zeros and replace them with the imputed values
        for col in cols_with_zeros:
            if not col in X_train.columns:
                continue
            X_train[col] = imputer.fit_transform(X_train[col].values.reshape(-1, 1))
            X_test[col] = imputer.transform(X_test[col].values.reshape(-1, 1))

    for col in categorical_cols:
        if not col in X_train.columns:
            continue
        X_train = pd.get_dummies(X_train, columns=[col], drop_first=True, dtype=int)
        X_test = pd.get_dummies(X_test, columns=[col], drop_first=True, dtype=int)

    results = []

    for model_cls in models:
        for scaler_cls in scalers:
            pp = base_pipeline(scaler_cls(), model_cls())
            fit_pipeline(pp, X_train, y_train)
            metrics = test_pipeline(pp, X_test, y_test)
            results.append(save_results(model_cls.__name__, scaler_cls.__name__, metrics))

    all_results = assemble_results(results)
    return all_results

Without any preprocessing

- Removed `District` (useless) and `Area` (deleted as PoC)

- Imputed `cols_with_zeros` and one-hot encoded `categorical_cols`

In [26]:
df = pd.read_csv('data.csv')
X = df.drop(columns=['Outcome', 'District', 'Area'])
y = df['Outcome']

run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)

accuracy  precision  recall
model                  scaler                                     
GaussianNB             MinMaxScaler         1.0        1.0     1.0
                       StandardScaler       1.0        1.0     1.0
KNeighborsClassifier   MinMaxScaler         1.0        1.0     1.0
                       StandardScaler       1.0        1.0     1.0
LogisticRegression     MinMaxScaler         1.0        1.0     1.0
                       StandardScaler       1.0        1.0     1.0
RandomForestClassifier MinMaxScaler         1.0        1.0     1.0
                       StandardScaler       1.0        1.0     1.0
SVC                    MinMaxScaler         1.0        1.0     1.0
                       StandardScaler       1.0        1.0     1.0

accuracy  precision  recall
scaler         model                                              
MinMaxScaler   GaussianNB                   1.0        1.0     1.0
               KNeighborsClassifier         1.0        1.0     1.0
               LogisticRegression           1.0        1.0     1.0
               RandomForestClassifier       1.0        1.0     1.0
               SVC                          1.0        1.0     1.0
StandardScaler GaussianNB                   1.0        1.0     1.0
               KNeighborsClassifier         1.0        1.0     1.0
               LogisticRegression           1.0        1.0     1.0
               RandomForestClassifier       1.0        1.0     1.0
               SVC                          1.0        1.0     1.0

Without data leakage columns

In [27]:
df = pd.read_csv('data.csv')
X = df.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM'])
y = df['Outcome']

run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)

accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000
KNeighborsClassifier   MinMaxScaler    0.993939   0.993007  0.994709
                       StandardScaler  1.000000   1.000000  1.000000
LogisticRegression     MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000
RandomForestClassifier MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000
SVC                    MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000

accuracy  precision    recall
scaler         model                                                
MinMaxScaler   GaussianNB              1.000000   1.000000  1.000000
               KNeighborsClassifier    0.993939   0.993007  0.994709
               LogisticRegression      1.000000   1.000000  1.000000
               RandomForestClassifier  1.000000   1.000000  1.000000
               SVC                     1.000000   1.000000  1.000000
StandardScaler GaussianNB              1.000000   1.000000  1.000000
               KNeighborsClassifier    1.000000   1.000000  1.000000
               LogisticRegression      1.000000   1.000000  1.000000
               RandomForestClassifier  1.000000   1.000000  1.000000
               SVC                     1.000000   1.000000  1.000000

Without clinical data

In [28]:
df = pd.read_csv('data.csv')
X = df.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count'])
y = df['Outcome']

print(X.columns)

run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)

Index(['Gender', 'Age', 'AreaType', 'HouseType', 'Fever_Duration',
       'Body_Temperature', 'Joint_Pain', 'Headache', 'Retro_Orbital_Pain',
       'Myalgia', 'Rash'],
      dtype='str')


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.996970   0.997368  0.996454
                       StandardScaler  0.996970   0.997368  0.996454
KNeighborsClassifier   MinMaxScaler    0.954545   0.951849  0.956715
                       StandardScaler  0.990909   0.990298  0.991163
LogisticRegression     MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000
RandomForestClassifier MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000
SVC                    MinMaxScaler    1.000000   1.000000  1.000000
                       StandardScaler  1.000000   1.000000  1.000000

accuracy  precision    recall
scaler         model                                                
MinMaxScaler   GaussianNB              0.996970   0.997368  0.996454
               KNeighborsClassifier    0.954545   0.951849  0.956715
               LogisticRegression      1.000000   1.000000  1.000000
               RandomForestClassifier  1.000000   1.000000  1.000000
               SVC                     1.000000   1.000000  1.000000
StandardScaler GaussianNB              0.996970   0.997368  0.996454
               KNeighborsClassifier    0.990909   0.990298  0.991163
               LogisticRegression      1.000000   1.000000  1.000000
               RandomForestClassifier  1.000000   1.000000  1.000000
               SVC                     1.000000   1.000000  1.000000

Without clinical data AND fever/body temp

In [14]:
df = pd.read_csv('data.csv')
X = df.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Fever_Duration', 'Body_Temperature'])
y = df['Outcome']

print(X.columns)

leakage_free_baseline = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
leakage_free_baseline[0]

Index(['Gender', 'Age', 'AreaType', 'HouseType', 'Joint_Pain', 'Headache',
       'Retro_Orbital_Pain', 'Myalgia', 'Rash'],
      dtype='str')


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.912121   0.908969  0.913374
                       StandardScaler  0.915152   0.911973  0.916920
LogisticRegression     MinMaxScaler    0.933333   0.930260  0.936395
                       StandardScaler  0.933333   0.930260  0.936395
RandomForestClassifier MinMaxScaler    0.933333   0.932615  0.930992
                       StandardScaler  0.930303   0.929135  0.928346
SVC                    MinMaxScaler    0.930303   0.929135  0.928346
                       StandardScaler  0.933333   0.931892  0.931892

In [35]:
from itertools import combinations
import numpy as np
import pandas as pd

def add_area_binning(df: pd.DataFrame, poverty_df: pd.DataFrame, n_bins: int = 5, single_col: bool = True) -> pd.DataFrame:
    out = df.copy()

    areas = (
        out[["Area", "Outcome"]]
        .dropna(subset=["Area"])
        .groupby("Area", as_index=False)
        .agg(n_patients=("Outcome", "size"))
        .merge(poverty_df[["Area", "HCR_upper_2022"]], on="Area", how="left")
        .sort_values(["HCR_upper_2022", "Area"], kind="mergesort")
        .reset_index(drop=True)
    )

    def find_balanced_cuts(sizes, n_bins=5):
        sizes = np.asarray(sizes, dtype=float)
        cumulative = np.concatenate([[0], sizes.cumsum()])
        target = sizes.sum() / n_bins

        best_score, best_bounds = None, None
        for cuts in combinations(range(1, len(sizes)), n_bins - 1):
            bounds = (0,) + cuts + (len(sizes),)
            counts = np.diff(cumulative[list(bounds)])
            deviations = np.abs(counts - target)
            score = (deviations.max(), (deviations ** 2).sum())
            if best_score is None or score < best_score:
                best_score, best_bounds = score, bounds

        return list(best_bounds), best_score

    bounds, _ = find_balanced_cuts(areas["n_patients"].values, n_bins=n_bins)

    inner_edges = [
        round(
            (areas.loc[b - 1, "HCR_upper_2022"] + areas.loc[b, "HCR_upper_2022"]) / 2,
            3,
        )
        for b in bounds[1:-1]
    ]

    bin_edges = [-np.inf] + inner_edges + [np.inf]
    bin_labels = [f"Q{i}" for i in range(1, n_bins + 1)]

    area_to_hcr = poverty_df.set_index("Area")["HCR_upper_2022"]
    out["Area_HCR"] = out["Area"].map(area_to_hcr)
    out["Area_bin"] = pd.cut(
        out["Area_HCR"],
        bins=bin_edges,
        labels=bin_labels,
        ordered=True,
    )

    if single_col:
        out['Area_bin_code'] = out['Area_bin'].cat.codes
        out = out.drop(columns=['Area_bin', 'Area_HCR'])
    else:
        out = pd.get_dummies(out, columns=['Area_bin'], drop_first=True, dtype=int)
        out = out.drop(columns=['Area_HCR'])


    return out

With separate columns

In [ ]:
df = pd.read_csv("data.csv")
poverty_df = pd.read_csv("area_poverty_table.csv")

df_with_bins = add_area_binning(df, poverty_df, n_bins=5, single_col=False)

# Then continue with the existing model setup
X = df_with_bins.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Body_Temperature', 'Fever_Duration'])
y = df_with_bins['Outcome']

sep_area_cols = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
print("Results:")
display(sep_area_cols[0])
print("Diff with leakage free baseline:")
sep_area_cols[0] - leakage_free_baseline[0]

Results:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.915152   0.913053  0.921423
                       StandardScaler  0.930303   0.927988  0.936452
LogisticRegression     MinMaxScaler    0.939394   0.936446  0.941686
                       StandardScaler  0.936364   0.933740  0.937240
RandomForestClassifier MinMaxScaler    0.930303   0.928491  0.929247
                       StandardScaler  0.924242   0.921819  0.923956
SVC                    MinMaxScaler    0.939394   0.938084  0.938084
                       StandardScaler  0.939394   0.936999  0.939885

Diff with leakage free baseline:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler    0.003030   0.004084  0.008049
                       StandardScaler  0.015152   0.016015  0.019532
LogisticRegression     MinMaxScaler    0.006061   0.006185  0.005291
                       StandardScaler  0.003030   0.003480  0.000844
RandomForestClassifier MinMaxScaler   -0.003030  -0.004124 -0.001745
                       StandardScaler -0.006061  -0.007316 -0.004390
SVC                    MinMaxScaler    0.009091   0.008949  0.009738
                       StandardScaler  0.006061   0.005107  0.007993

With one column

In [ ]:
df = pd.read_csv("data.csv")
poverty_df = pd.read_csv("area_poverty_table.csv")

df_with_bins = add_area_binning(df, poverty_df, n_bins=5, single_col=True)

# Then continue with the existing model setup
X = df_with_bins.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Body_Temperature', 'Fever_Duration'])
y = df_with_bins['Outcome']

run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
one_area_col = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
print("Results:")
display(one_area_col[0])
print("Diff with leakage free baseline:")
display(one_area_col[0] - leakage_free_baseline[0])
print("Diff with sep area cols:")
display(one_area_col[0] - sep_area_cols[0])

Results:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.915152   0.912395  0.915119
                       StandardScaler  0.912121   0.908889  0.914274
LogisticRegression     MinMaxScaler    0.942424   0.939444  0.945232
                       StandardScaler  0.939394   0.936446  0.941686
RandomForestClassifier MinMaxScaler    0.939394   0.937474  0.938985
                       StandardScaler  0.933333   0.931304  0.932793
SVC                    MinMaxScaler    0.930303   0.929135  0.928346
                       StandardScaler  0.927273   0.925134  0.926601

Diff with leakage free baseline:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler    0.003030   0.003426  0.001745
                       StandardScaler -0.003030  -0.003084 -0.002646
LogisticRegression     MinMaxScaler    0.009091   0.009184  0.008837
                       StandardScaler  0.006061   0.006185  0.005291
RandomForestClassifier MinMaxScaler    0.006061   0.004859  0.007993
                       StandardScaler  0.003030   0.002169  0.004447
SVC                    MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler -0.006061  -0.006759 -0.005291

Diff with sep area cols:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler    0.000000  -0.000658 -0.006304
                       StandardScaler -0.018182  -0.019099 -0.022177
LogisticRegression     MinMaxScaler    0.003030   0.002999  0.003546
                       StandardScaler  0.003030   0.002705  0.004447
RandomForestClassifier MinMaxScaler    0.009091   0.008983  0.009738
                       StandardScaler  0.009091   0.009485  0.008837
SVC                    MinMaxScaler   -0.009091  -0.008949 -0.009738
                       StandardScaler -0.012121  -0.011865 -0.013284

In [42]:
def add_age_binning(df: pd.DataFrame, balance_tolerance: float = 0.10, single_col: bool = True) -> pd.DataFrame:
    """Add Age_bin using the same balanced-split logic from Feature_Engineering.ipynb."""
    out = df.copy()

    age_counts = (
        out.groupby("Age")
        .agg(n_patients=("Outcome", "size"), positive_rate=("Outcome", "mean"))
        .sort_index()
    )

    def enumerate_splits(n_groups, n_bins):
        return np.array(
            [(0,) + cuts + (n_groups,) for cuts in combinations(range(1, n_groups), n_bins - 1)],
            dtype=np.int32,
        )

    def score_splits(splits, group_sizes, group_positives):
        group_sizes = np.asarray(group_sizes, dtype=float)
        group_positives = np.asarray(group_positives, dtype=float)
        cum_n = np.concatenate([[0], group_sizes.cumsum()])
        cum_p = np.concatenate([[0], group_positives.cumsum()])

        n = np.diff(cum_n[splits], axis=1)
        p = np.diff(cum_p[splits], axis=1)
        total_n, total_p = group_sizes.sum(), group_positives.sum()
        target = total_n / (splits.shape[1] - 1)

        expected_pos = n * total_p / total_n
        expected_neg = n * (total_n - total_p) / total_n
        with np.errstate(divide="ignore", invalid="ignore"):
            chi2 = np.nan_to_num(
                (p - expected_pos) ** 2 / expected_pos + ((n - p) - expected_neg) ** 2 / expected_neg
            ).sum(axis=1)

        rates = np.divide(p, n, out=np.full_like(p, np.nan), where=n > 0)
        return {
            "n": n,
            "rates": rates,
            "balance_deviation": np.abs(n - target).max(axis=1),
            "rate_spread": np.nanmax(rates, axis=1) - np.nanmin(rates, axis=1),
            "chi2": chi2,
        }

    age_values = age_counts.index.to_numpy()
    age_sizes = age_counts["n_patients"].to_numpy()
    age_positives = out.groupby("Age")["Outcome"].sum().sort_index().to_numpy()

    splits = enumerate_splits(len(age_values), 5)
    scores = score_splits(splits, age_sizes, age_positives)

    tol = balance_tolerance * len(out) / 5
    allowed = np.where(scores["balance_deviation"] <= tol)[0]
    if len(allowed) == 0:
        raise ValueError("No age split satisfies the balance tolerance.")

    chosen = allowed[scores["chi2"][allowed].argmax()]
    age_bounds = splits[chosen]

    age_edges = [-np.inf] + [(age_values[b - 1] + age_values[b]) / 2 for b in age_bounds[1:-1]] + [np.inf]
    age_labels = [
        f"{age_values[age_bounds[i]]}-{age_values[age_bounds[i + 1] - 1]}"
        for i in range(5)
    ]

    out["Age_bin"] = pd.cut(out["Age"], bins=age_edges, labels=age_labels, ordered=True)
    if single_col:
        out['Age_bin_code'] = out['Age_bin'].cat.codes
        out = out.drop(columns=['Age_bin'])
    else:
        out = pd.get_dummies(out, columns=['Age_bin'], drop_first=True, dtype=int)


    return out

With one col area binning + one col age binning

In [40]:
df = pd.read_csv("data.csv")
poverty_df = pd.read_csv("area_poverty_table.csv")

df_with_bins = add_area_binning(df, poverty_df, n_bins=5, single_col=True)
df_with_bins = add_age_binning(df_with_bins, balance_tolerance=0.10, single_col=True)

# Then continue with the existing model setup
X = df_with_bins.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Body_Temperature', 'Fever_Duration'])
y = df_with_bins['Outcome']

one_age_col = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
print("Results:")
display(one_age_col[0])
print("Diff with one area column:")
one_age_col[0] - one_area_col[0]


Results:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.909091   0.905842  0.911629
                       StandardScaler  0.921212   0.918256  0.922211
LogisticRegression     MinMaxScaler    0.942424   0.939444  0.945232
                       StandardScaler  0.939394   0.936446  0.941686
RandomForestClassifier MinMaxScaler    0.936364   0.934672  0.935438
                       StandardScaler  0.939394   0.938084  0.938084
SVC                    MinMaxScaler    0.930303   0.928491  0.929247
                       StandardScaler  0.930303   0.927598  0.931048

Diff with one area column:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler   -0.006061  -0.006553 -0.003490
                       StandardScaler  0.009091   0.009367  0.007937
LogisticRegression     MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
RandomForestClassifier MinMaxScaler    0.003030   0.003368  0.002646
                       StandardScaler  0.009091   0.009593  0.008837
SVC                    MinMaxScaler    0.000000  -0.000644  0.000901
                       StandardScaler  0.003030   0.002465  0.004447

With one col area binning + one-hot age binning

In [43]:
df = pd.read_csv("data.csv")
poverty_df = pd.read_csv("area_poverty_table.csv")

df_with_bins = add_area_binning(df, poverty_df, n_bins=5, single_col=True)
df_with_bins = add_age_binning(df_with_bins, balance_tolerance=0.10, single_col=False)

# Then continue with the existing model setup
X = df_with_bins.drop(columns=['Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Body_Temperature', 'Fever_Duration'])
y = df_with_bins['Outcome']

one_hot_age_col = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
print("Results:")
display(one_hot_age_col[0])
print("Diff with one area column:")
display(one_hot_age_col[0] - one_area_col[0])


Results:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.906061   0.903003  0.909884
                       StandardScaler  0.884848   0.884023  0.892266
LogisticRegression     MinMaxScaler    0.942424   0.939444  0.945232
                       StandardScaler  0.945455   0.942564  0.947878
RandomForestClassifier MinMaxScaler    0.936364   0.934140  0.936339
                       StandardScaler  0.936364   0.933740  0.937240
SVC                    MinMaxScaler    0.927273   0.924389  0.928403
                       StandardScaler  0.924242   0.921221  0.925757

Diff with one area column:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler   -0.009091  -0.009392 -0.005235
                       StandardScaler -0.027273  -0.024866 -0.022008
LogisticRegression     MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.006061   0.006118  0.006192
RandomForestClassifier MinMaxScaler    0.003030   0.002836  0.003546
                       StandardScaler  0.006061   0.005249  0.007993
SVC                    MinMaxScaler   -0.003030  -0.004746  0.000056
                       StandardScaler -0.003030  -0.003913 -0.000844

In [44]:
print("one age col vs one hot age col:")
display(one_age_col[0] - one_hot_age_col[0])

one age col vs one hot age col:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler    0.003030   0.002839  0.001745
                       StandardScaler  0.036364   0.034232  0.029945
LogisticRegression     MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler -0.006061  -0.006118 -0.006192
RandomForestClassifier MinMaxScaler    0.000000   0.000532 -0.000901
                       StandardScaler  0.003030   0.004344  0.000844
SVC                    MinMaxScaler    0.003030   0.004102  0.000844
                       StandardScaler  0.006061   0.006378  0.005291

With one col area binning w/o area hcp + one col age binning w/o age

In [46]:
df = pd.read_csv("data.csv")
poverty_df = pd.read_csv("area_poverty_table.csv")

df_with_bins = add_area_binning(df, poverty_df, n_bins=5, single_col=True)
df_with_bins = add_age_binning(df_with_bins, balance_tolerance=0.10, single_col=True)

# Then continue with the existing model setup
X = df_with_bins.drop(columns=['Age', 'Outcome', 'District', 'Area', 'NS1', 'IgG', 'IgM', 'WBC_Count', 'Platelet_Count', 'Body_Temperature', 'Fever_Duration'])
y = df_with_bins['Outcome']


final_run = run_pipeline(X, y, imputer=IterativeImputer(n_nearest_features=5), with_logs=True)
print("Results:")
display(final_run[0])
print("Diff with baseline:")
display(final_run[0] - leakage_free_baseline[0])


Results:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.927273   0.927234  0.923900
                       StandardScaler  0.927273   0.927234  0.923900
KNeighborsClassifier   MinMaxScaler    0.918182   0.915095  0.919565
                       StandardScaler  0.912121   0.908930  0.915175
LogisticRegression     MinMaxScaler    0.942424   0.939444  0.945232
                       StandardScaler  0.939394   0.936446  0.941686
RandomForestClassifier MinMaxScaler    0.933333   0.931892  0.931892
                       StandardScaler  0.930303   0.928491  0.929247
SVC                    MinMaxScaler    0.927273   0.925701  0.925701
                       StandardScaler  0.930303   0.927979  0.930147

Diff with baseline:


accuracy  precision    recall
model                  scaler                                       
GaussianNB             MinMaxScaler    0.000000   0.000000  0.000000
                       StandardScaler  0.000000   0.000000  0.000000
KNeighborsClassifier   MinMaxScaler    0.006061   0.006126  0.006192
                       StandardScaler -0.003030  -0.003043 -0.001745
LogisticRegression     MinMaxScaler    0.009091   0.009184  0.008837
                       StandardScaler  0.006061   0.006185  0.005291
RandomForestClassifier MinMaxScaler    0.000000  -0.000723  0.000901
                       StandardScaler  0.000000  -0.000644  0.000901
SVC                    MinMaxScaler   -0.003030  -0.003435 -0.002646
                       StandardScaler -0.003030  -0.003913 -0.001745